# World Cup 2026 — Player Statistics Pipeline

Run the cells **in order** the first time. After that you can re-run any single cell independently.

| Cell | What it does |
|------|--------------|
| 1. Configuration | Set your API key, season, leagues |
| 2. Imports | Load all modules |
| 3. Init DB | Create the DuckDB schema |
| 4. Fetch Teams | Download 32 World Cup nations |
| 5. Fetch Squads | Download squad rosters |
| 6. Fetch Match Stats | Download per-match player stats |


## Cell 1 — Configuration
Edit the values here before running anything else.

In [ ]:
# ── Your API-Football key from https://api-sports.io ─────────────────────────
API_FOOTBALL_KEY = "your_api_key_here"

# ── Local database file ───────────────────────────────────────────────────────
DB_PATH = "playerstats.db"

# ── Club season to fetch match stats for ─────────────────────────────────────
SEASON = 2025

# ── Leagues to fetch  (remove any you don't need) ────────────────────────────
# 39=Premier League  140=La Liga  78=Bundesliga  135=Serie A  61=Ligue 1
# 2=Champions League  88=Eredivisie  94=Primeira Liga  203=Süper Lig
LEAGUES = [39, 140, 78, 135, 61]

# ── Rate limit (requests per minute) ─────────────────────────────────────────
# Free tier = 10/min.  Paid tiers can go higher.
REQUESTS_PER_MINUTE = 10

print("Configuration set.")
print(f"  Season  : {SEASON}")
print(f"  Leagues : {LEAGUES}")
print(f"  DB path : {DB_PATH}")

## Cell 2 — Imports

In [ ]:
import logging
import os

from rich.console import Console
from rich.logging import RichHandler
from rich.panel import Panel
from rich.table import Table

from src.api.client import FootballAPIClient
from src.db.database import Database
from src.etl.extract import Extractor
from src.etl.load import Loader
from src.etl.transform import (
    enrich_players_from_stats,
    generate_date_dimension,
    transform_competitions_from_fixtures,
    transform_match_player_stats,
    transform_matches,
    transform_players,
    transform_teams,
    transform_teams_from_fixtures,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(message)s",
    handlers=[RichHandler(rich_tracebacks=True, show_path=False)],
)
logger = logging.getLogger("pipeline")
console = Console()


def _make_db():
    return Database(DB_PATH)


def _make_client():
    if not API_FOOTBALL_KEY or API_FOOTBALL_KEY == "your_api_key_here":
        raise ValueError("Set API_FOOTBALL_KEY in Cell 1 before running.")
    return FootballAPIClient(api_key=API_FOOTBALL_KEY, requests_per_minute=REQUESTS_PER_MINUTE)


def _summary(title, data):
    table = Table(title=title, show_header=False, expand=False)
    table.add_column("Key", style="cyan")
    table.add_column("Value", style="white")
    for k, v in data.items():
        table.add_row(str(k), str(v))
    console.print(table)


print("Imports OK.")

## Cell 3 — Initialise Database

Creates all tables and views in the local DuckDB file and populates the date dimension.
Safe to re-run — uses `CREATE TABLE IF NOT EXISTS`.

In [ ]:
console.print(Panel("[bold]Initialising database schema[/bold]", style="blue"))

with _make_db() as db:
    db.initialize_schema()
    dates = generate_date_dimension(start_year=2024, end_year=2027)
    loader = Loader(db)
    n = loader.load_date_dimension(dates)

console.print(f"[green]Done.[/green] Schema ready — {n} date rows loaded into dim_date.")

## Cell 4 — Fetch World Cup Teams

Downloads the 32 national teams for the 2026 FIFA World Cup and loads them into `dim_team`.

**API calls:** 1

In [ ]:
WC_SEASON = 2026

console.print(Panel(f"[bold]Fetching World Cup {WC_SEASON} teams[/bold]", style="blue"))

client = _make_client()
extractor = Extractor(client)
raw_teams = extractor.extract_world_cup_teams(season=WC_SEASON)
team_rows = transform_teams(raw_teams)

with _make_db() as db:
    loader = Loader(db)
    n = loader.upsert_teams(team_rows)

_summary("fetch-teams summary", {
    "Teams fetched": len(raw_teams),
    "Rows upserted": n,
    "API calls used": client._request_count,
})

## Cell 5 — Fetch Squad Rosters

Downloads the player roster for every World Cup team loaded in Cell 4 and stores them in `dim_player`.

**API calls:** 1 per team (~32 calls total)

In [ ]:
console.print(Panel("[bold]Fetching World Cup squads[/bold]", style="blue"))

with _make_db() as db:
    teams_df = db.query("SELECT api_team_id, name FROM dim_team WHERE is_world_cup_2026 = TRUE")

if teams_df.empty:
    console.print("[yellow]No teams in DB — run Cell 4 first.[/yellow]")
else:
    raw_teams_stub = [
        {"team": {"id": int(row["api_team_id"]), "name": row["name"]}}
        for _, row in teams_df.iterrows()
    ]

    client = _make_client()
    extractor = Extractor(client)
    raw_squads = extractor.extract_world_cup_squads(teams=raw_teams_stub)

    with _make_db() as db:
        loader = Loader(db)
        team_id_map = loader.get_team_id_map()
        player_rows = transform_players(raw_squads, wc_team_map=team_id_map)
        n = loader.upsert_players(player_rows)

    _summary("fetch-squads summary", {
        "Teams processed": len(raw_squads),
        "Players upserted": n,
        "API calls used": client._request_count,
    })

## Cell 6 — Fetch Per-Match Player Statistics

For each league in `LEAGUES` (configured in Cell 1):
1. Downloads the full fixture list for `SEASON`
2. For every completed match, fetches individual player stats
3. Loads everything into `dim_match` and `fact_player_match_stats`

**API calls:** ~2–3 to get the fixture list + **1 call per completed match**.  
A full Premier League season (~380 matches) costs ~382 calls.  
Free tier is 100 calls/day — consider running one league at a time.

> To run a single league, change `LEAGUES` in Cell 1 or override below.

In [ ]:
# Override leagues for this run if needed, e.g. leagues_this_run = [39]
leagues_this_run = LEAGUES

console.print(
    Panel(
        f"[bold]Fetching per-match stats (season {SEASON})[/bold]\n"
        f"Leagues: {leagues_this_run}\n"
        "[dim]1 API call per completed match[/dim]",
        style="blue",
    )
)

client = _make_client()
extractor = Extractor(client)
enriched_fixtures = extractor.extract_match_stats(league_ids=leagues_this_run, season=SEASON)

if not enriched_fixtures:
    console.print("[yellow]No completed fixtures found.[/yellow]")
else:
    with _make_db() as db:
        loader = Loader(db)

        # Competitions
        comp_rows = transform_competitions_from_fixtures(enriched_fixtures)
        loader.upsert_competitions(comp_rows)

        # Club teams
        team_rows = transform_teams_from_fixtures(enriched_fixtures)
        existing_map = loader.get_team_id_map()
        new_teams = [t for t in team_rows if t["api_team_id"] not in existing_map]
        if new_teams:
            loader.upsert_teams(new_teams)

        team_id_map = loader.get_team_id_map()
        comp_id_map = loader.get_competition_id_map()

        # Matches
        match_rows = transform_matches(enriched_fixtures, team_id_map, comp_id_map)
        n_matches = loader.upsert_matches(match_rows)

        match_id_map = loader.get_match_id_map()
        player_id_map = loader.get_player_id_map()
        national_team_map = loader.get_national_team_map()

        # Fact rows
        fact_rows = transform_match_player_stats(
            enriched_fixtures,
            player_id_map=player_id_map,
            match_id_map=match_id_map,
            team_id_map=team_id_map,
            competition_id_map=comp_id_map,
            national_team_map=national_team_map,
        )
        n_facts = loader.upsert_match_player_stats(fact_rows)

    _summary("fetch-match-stats summary", {
        "Season": SEASON,
        "Leagues": len(leagues_this_run),
        "Fixtures processed": len(enriched_fixtures),
        "Matches upserted": n_matches,
        "Fact rows upserted": n_facts,
        "API calls used": client._request_count,
    })

## Cell 7 — Quick DB Check (optional)

Run this at any time to see how much data is in the database.

In [ ]:
with _make_db() as db:
    counts = {
        "dim_team": db.query("SELECT COUNT(*) AS n FROM dim_team").iloc[0]["n"],
        "dim_player": db.query("SELECT COUNT(*) AS n FROM dim_player").iloc[0]["n"],
        "dim_match": db.query("SELECT COUNT(*) AS n FROM dim_match").iloc[0]["n"],
        "dim_competition": db.query("SELECT COUNT(*) AS n FROM dim_competition").iloc[0]["n"],
        "fact_player_match_stats": db.query("SELECT COUNT(*) AS n FROM fact_player_match_stats").iloc[0]["n"],
    }

_summary("Row counts", counts)